In [1]:
# Part 1: Kaggle environment setup
!pip install -q \
bitsandbytes==0.46.1 \
transformers==4.53.3 \
accelerate==1.8.1 \
scipy==1.16.1 \
num2words==0.5.14 \
pyyaml==6.0.2 \
pandas==2.2.2 \
matplotlib==3.10.0 \
hf_transfer==0.1.9 \
peft==0.17.0 \
torchaudio \
openai-whisper

import os
import sys
from pathlib import Path

BASE_WORK_DIR = Path("/kaggle/working/nbmmoshi")
HF_CACHE_DIR = BASE_WORK_DIR / "hf_cache"
MOSHI_REPO_DIR = BASE_WORK_DIR / "moshi_repo"

BASE_WORK_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

if not MOSHI_REPO_DIR.exists():
    !git clone https://github.com/kyutai-labs/moshi.git {MOSHI_REPO_DIR}

sys.path.append(str(MOSHI_REPO_DIR))

print("Kaggle environment is ready.")
print(f"Working directory: {BASE_WORK_DIR}")
print(f"Hugging Face cache: {HF_CACHE_DIR}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 47.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 24.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 102.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 59.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 83.4 MB/s eta 

In [2]:
# Optional check: confirm bitsandbytes version after install
import bitsandbytes as bnb
print("bitsandbytes version:", bnb.__version__)


bitsandbytes version: 0.46.1


In [3]:
# Part 2: Resume from previous checkpoint and upload current checkpoint dataset
import json
import re
import shutil
import subprocess
import time
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_WORK_DIR = Path("/kaggle/working/nbmmoshi")
INPUT_FILE = "/kaggle/input/datasets/thedevastator/grade-school-math-8k-q-a/main_train.csv"
OUTPUT_DIR = BASE_WORK_DIR / "cot_generated"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

validated_path = OUTPUT_DIR / "cot_validated.jsonl"
rejected_path = OUTPUT_DIR / "cot_rejected.jsonl"
validated_partial_path = OUTPUT_DIR / "cot_validated_partial.jsonl"
rejected_partial_path = OUTPUT_DIR / "cot_rejected_partial.jsonl"

SAVE_INTERVAL_SECONDS = 10 * 3600
PRINT_INTERVAL_ROWS = 10

KAGGLE_USERNAME = "nbmproject"

# Change only this value each run:
CURRENT_CHECKPOINT_NUMBER = 5

PREV_CHECKPOINT_NUMBER = CURRENT_CHECKPOINT_NUMBER - 1

if PREV_CHECKPOINT_NUMBER == 1:
    PREV_CHECKPOINT_DIR = Path("/kaggle/input/datasets/nbmproject/nbmmoshi-cot-checkpoints")
else:
    PREV_CHECKPOINT_DIR = Path(
        f"/kaggle/input/datasets/nbmproject/nbmmoshi-cot-checkpoint-{PREV_CHECKPOINT_NUMBER}"
    )

DATASET_SLUG = f"nbmmoshi-cot-checkpoint-{CURRENT_CHECKPOINT_NUMBER}"
DATASET_ID = f"{KAGGLE_USERNAME}/{DATASET_SLUG}"
UPLOAD_DIR = BASE_WORK_DIR / f"kaggle_checkpoint_{CURRENT_CHECKPOINT_NUMBER}_upload"

print(f"Current checkpoint number: {CURRENT_CHECKPOINT_NUMBER}")
print(f"Previous checkpoint number: {PREV_CHECKPOINT_NUMBER}")
print(f"Previous checkpoint dir: {PREV_CHECKPOINT_DIR}")
print(f"Current dataset id: {DATASET_ID}")

print(f"Loading data from {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)

print("CSV columns:", list(df.columns))
print("Total rows:", len(df))

PROMPT_TEMPLATE = """Solve this math problem step by step. Show your reasoning clearly, then give the final numeric answer.
IMPORTANT FORMAT:
- First, show your step-by-step reasoning under REASONING:
- Then give ONLY the final concise answer under ANSWER: (one short sentence with the number)

Question: {question}
REASONING:"""


def extract_number(text: str):
    numbers = re.findall(r"-?\d+\.?\d*", text.replace(",", ""))
    return numbers[-1] if numbers else None


def read_jsonl(path: Path):
    items = []
    if not path.exists():
        return items
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items


def save_jsonl(path: Path, items):
    with open(path, "w", encoding="utf-8") as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


def load_previous_checkpoint():
    validated = read_jsonl(PREV_CHECKPOINT_DIR / "cot_validated_partial.jsonl")
    rejected = read_jsonl(PREV_CHECKPOINT_DIR / "cot_rejected_partial.jsonl")

    if not validated and not rejected:
        raise FileNotFoundError(
            f"Could not find checkpoint partial files in {PREV_CHECKPOINT_DIR}. "
            "Make sure the previous checkpoint dataset is added as Kaggle input."
        )

    return validated, rejected


def prepare_upload_dir(current_idx, validated, rejected, final_save=False):
    if UPLOAD_DIR.exists():
        shutil.rmtree(UPLOAD_DIR)
    UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

    shutil.copy2(validated_partial_path, UPLOAD_DIR / validated_partial_path.name)
    shutil.copy2(rejected_partial_path, UPLOAD_DIR / rejected_partial_path.name)

    if final_save:
        shutil.copy2(validated_path, UPLOAD_DIR / validated_path.name)
        shutil.copy2(rejected_path, UPLOAD_DIR / rejected_path.name)

    metadata = {
        "title": f"NBM Moshi CoT Checkpoint {CURRENT_CHECKPOINT_NUMBER}",
        "id": DATASET_ID,
        "licenses": [{"name": "CC0-1.0"}],
    }
    with open(UPLOAD_DIR / "dataset-metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    manifest = f"""# NBM Moshi CoT Checkpoint {CURRENT_CHECKPOINT_NUMBER}

previous_checkpoint_number: {PREV_CHECKPOINT_NUMBER}
current_checkpoint_number: {CURRENT_CHECKPOINT_NUMBER}
row_processed: {current_idx + 1}
validated_count: {len(validated)}
rejected_count: {len(rejected)}
final_save: {final_save}
source_csv: {INPUT_FILE}
model_name: {MODEL_NAME}
saved_at_unix: {time.time()}
"""
    with open(UPLOAD_DIR / "MANIFEST.md", "w", encoding="utf-8") as f:
        f.write(manifest)


def upload_checkpoint(current_idx, validated, rejected, final_save=False):
    prepare_upload_dir(current_idx, validated, rejected, final_save=final_save)

    message = (
        f"Final upload for checkpoint {CURRENT_CHECKPOINT_NUMBER} at row {current_idx + 1}"
        if final_save
        else f"Checkpoint {CURRENT_CHECKPOINT_NUMBER} upload at row {current_idx + 1}"
    )

    print(f"\nUploading dataset: {DATASET_ID}")

    create_result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(UPLOAD_DIR), "-u"],
        capture_output=True,
        text=True,
    )

    if create_result.returncode == 0:
        print(create_result.stdout or "(no stdout)")
        print(f"Dataset created: https://www.kaggle.com/datasets/{DATASET_ID}")
        return

    print(f"Create failed (rc={create_result.returncode}), trying version update...")
    if create_result.stderr:
        print(create_result.stderr)

    version_result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(UPLOAD_DIR), "-m", message],
        capture_output=True,
        text=True,
    )

    print(version_result.stdout or "(no stdout)")
    if version_result.returncode != 0:
        print("Version upload failed.")
        if version_result.stderr:
            print(version_result.stderr)
    else:
        print(f"Dataset updated: https://www.kaggle.com/datasets/{DATASET_ID}")


def save_progress(validated, rejected, current_idx, final_save=False):
    save_jsonl(validated_partial_path, validated)
    save_jsonl(rejected_partial_path, rejected)

    if final_save:
        save_jsonl(validated_path, validated)
        save_jsonl(rejected_path, rejected)

    label = "FINAL SAVE" if final_save else "CHECKPOINT SAVE"
    print(
        f"\n[{label}] checkpoint={CURRENT_CHECKPOINT_NUMBER} | row={current_idx + 1} | "
        f"validated={len(validated)} | rejected={len(rejected)}"
    )
    print(f"  Partial validated: {validated_partial_path}")
    print(f"  Partial rejected : {rejected_partial_path}")
    if final_save:
        print(f"  Final validated  : {validated_path}")
        print(f"  Final rejected   : {rejected_path}")

    upload_checkpoint(current_idx, validated, rejected, final_save=final_save)


print("Loading previous checkpoint...")
validated, rejected = load_previous_checkpoint()
start_idx = len(validated) + len(rejected)

save_jsonl(validated_partial_path, validated)
save_jsonl(rejected_partial_path, rejected)

print(f"Recovered validated={len(validated)} | rejected={len(rejected)}")
print(f"Resuming from row index: {start_idx}")
print("First resumed row id will be:", f"train_{start_idx}")
print(f"Next dataset will be: https://www.kaggle.com/datasets/{DATASET_ID}")

print("Loading Qwen model (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

start_time = time.time()
last_save_time = start_time

for idx in range(start_idx, len(df)):
    row = df.iloc[idx]

    question = str(row["question"])
    raw_answer = str(row["answer"])

    gt_match = re.search(r"####\s*(-?\d+\.?\d*)", raw_answer)
    ground_truth = gt_match.group(1) if gt_match else extract_number(raw_answer)

    prompt = PROMPT_TEMPLATE.format(question=question)
    messages = [
        {
            "role": "system",
            "content": "You are a precise math tutor. Always show step-by-step reasoning, then a concise final answer.",
        },
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    ans_match = re.search(r"ANSWER:\s*(.+?)$", response, re.DOTALL | re.IGNORECASE)
    if ans_match:
        concise_ans = ans_match.group(1).strip().split("\n")[0]
        reasoning = re.sub(
            r"^REASONING:\s*",
            "",
            response[:ans_match.start()].strip(),
            flags=re.IGNORECASE,
        )
    else:
        reasoning = response
        concise_ans = ""

    predicted_num = extract_number(concise_ans)
    is_correct = (
        predicted_num is not None
        and ground_truth is not None
        and abs(float(predicted_num) - float(ground_truth)) < 1e-2
    )

    result = {
        "id": f"train_{idx}",
        "question": question,
        "ground_truth": ground_truth,
        "generated_reasoning": reasoning,
        "generated_concise_answer": concise_ans,
        "is_correct": is_correct,
    }

    if is_correct:
        validated.append(result)
    else:
        rejected.append(result)

    if (idx + 1) % PRINT_INTERVAL_ROWS == 0:
        elapsed_hours = (time.time() - start_time) / 3600
        print(
            f"[{idx + 1}/{len(df)}] Validated: {len(validated)} | "
            f"Rejected: {len(rejected)} | Elapsed this session: {elapsed_hours:.2f} hr"
        )

    if time.time() - last_save_time >= SAVE_INTERVAL_SECONDS:
        save_progress(validated, rejected, idx, final_save=False)
        last_save_time = time.time()

save_progress(validated, rejected, len(df) - 1, final_save=True)

print("\nPart 2 complete.")
print(f"Validated samples: {len(validated)}")
print(f"Rejected samples: {len(rejected)}")
print(f"Saved validated file to: {validated_path}")
print(f"Saved rejected file to: {rejected_path}")
print(f"Current checkpoint dataset: https://www.kaggle.com/datasets/{DATASET_ID}")


Current checkpoint number: 5
Previous checkpoint number: 4
Previous checkpoint dir: /kaggle/input/datasets/nbmproject/nbmmoshi-cot-checkpoint-4
Current dataset id: nbmproject/nbmmoshi-cot-checkpoint-5
Loading data from /kaggle/input/datasets/thedevastator/grade-school-math-8k-q-a/main_train.csv...
CSV columns: ['question', 'answer']
Total rows: 7473
Loading previous checkpoint...
Recovered validated=2419 | rejected=338
Resuming from row index: 2757
First resumed row id will be: train_2757
Next dataset will be: https://www.kaggle.com/datasets/nbmproject/nbmmoshi-cot-checkpoint-5
Loading Qwen model (4-bit)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[2760/7473] Validated: 2422 | Rejected: 338 | Elapsed this session: 0.02 hr
[2770/7473] Validated: 2432 | Rejected: 338 | Elapsed this session: 0.07 hr
[2780/7473] Validated: 2441 | Rejected: 339 | Elapsed this session: 0.11 hr
[2790/7473] Validated: 2451 | Rejected: 339 | Elapsed this session: 0.17 hr
[2800/7473] Validated: 2460 | Rejected: 340 | Elapsed this session: 0.23 hr
[2810/7473] Validated: 2470 | Rejected: 340 | Elapsed this session: 0.27 hr
[2820/7473] Validated: 2479 | Rejected: 341 | Elapsed this session: 0.33 hr
[2830/7473] Validated: 2488 | Rejected: 342 | Elapsed this session: 0.38 hr
[2840/7473] Validated: 2495 | Rejected: 345 | Elapsed this session: 0.43 hr
[2850/7473] Validated: 2503 | Rejected: 347 | Elapsed this session: 0.48 hr
[2860/7473] Validated: 2512 | Rejected: 348 | Elapsed this session: 0.54 hr
[2870/7473] Validated: 2521 | Rejected: 349 | Elapsed this session: 0.59 hr
[2880/7473] Validated: 2530 | Rejected: 350 | Elapsed this session: 0.64 hr
[2890/7473] 

KeyboardInterrupt: 

In [ ]:
# Part 3: Paraphrase the validated questions
import json
from pathlib import Path

BASE_WORK_DIR = Path("/kaggle/working/nbmmoshi")
INPUT_FILE = BASE_WORK_DIR / "cot_generated" / "cot_validated.jsonl"
OUTPUT_DIR = BASE_WORK_DIR / "augmented"
PARAPHRASES_PER_Q = 1

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PARAPHRASE_PROMPT = """Rephrase this math word problem. Keep the EXACT same numbers and mathematical relationship, but change the names, context, and wording. The answer must remain identical.

Original: {question}

Rephrased version:"""


def paraphrase_question(question: str) -> str:
    messages = [
        {
            "role": "system",
            "content": "You rephrase math problems. Keep all numbers and math relationships identical. Change names and context only. Output ONLY the rephrased problem, nothing else.",
        },
        {"role": "user", "content": PARAPHRASE_PROMPT.format(question=question)},
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
    return response.strip().split("\n")[0]


print("Loading validated data...")
with open(INPUT_FILE, encoding="utf-8") as f:
    validated = [json.loads(line) for line in f]

augmented = []
for item in validated:
    item["source"] = "original"
    augmented.append(item)

success = 0
for i, item in enumerate(validated):
    for p in range(PARAPHRASES_PER_Q):
        try:
            new_q = paraphrase_question(item["question"])
            if len(new_q) < 20 or new_q == item["question"]:
                continue

            augmented.append(
                {
                    "id": f"{item['id']}_para{p}",
                    "question": new_q,
                    "ground_truth": item["ground_truth"],
                    "generated_reasoning": item["generated_reasoning"],
                    "generated_concise_answer": item["generated_concise_answer"],
                    "source": "paraphrase",
                }
            )
            success += 1
        except Exception:
            pass

    if (i + 1) % 100 == 0:
        print(f"[{i + 1}/{len(validated)}] Paraphrases generated: {success}")

with open(OUTPUT_DIR / "augmented_full.jsonl", "w", encoding="utf-8") as f:
    for item in augmented:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Part 3 complete. Output saved to {OUTPUT_DIR}")


In [ ]:
# Part 4: TTS audio generation
import json
import os
import re
import site
import sys
from pathlib import Path

import numpy as np
from scipy.io import wavfile


def patch_stream_generator():
    stream_gen_path = None
    for sp in site.getsitepackages() + sys.path:
        candidate = os.path.join(sp, "TTS/tts/layers/xtts/stream_generator.py")
        if os.path.exists(candidate):
            stream_gen_path = candidate
            break

    if stream_gen_path is None:
        print("Could not find stream_generator.py, skipping patch")
        return

    with open(stream_gen_path, "r", encoding="utf-8") as f:
        content = f.read()

    if "except ImportError" in content:
        print("stream_generator.py already patched")
        return

    old_import = (
        "from transformers import (\n"
        "    BeamSearchScorer,\n"
        "    ConstrainedBeamSearchScorer,\n"
        "    DisjunctiveConstraint,\n"
        "    GenerationConfig,\n"
        "    GenerationMixin,\n"
        "    LogitsProcessorList,\n"
        "    PhrasalConstraint,\n"
        "    PreTrainedModel,\n"
        "    StoppingCriteriaList,\n"
        ")\n"
        "from transformers.generation.utils import GenerateOutput, SampleOutput, logger"
    )

    new_import = (
        "from transformers import (\n"
        "    GenerationConfig,\n"
        "    GenerationMixin,\n"
        "    LogitsProcessorList,\n"
        "    PreTrainedModel,\n"
        "    StoppingCriteriaList,\n"
        ")\n"
        "try:\n"
        "    from transformers import (\n"
        "        BeamSearchScorer,\n"
        "        ConstrainedBeamSearchScorer,\n"
        "        DisjunctiveConstraint,\n"
        "        PhrasalConstraint,\n"
        "    )\n"
        "except ImportError:\n"
        "    BeamSearchScorer = None\n"
        "    ConstrainedBeamSearchScorer = None\n"
        "    DisjunctiveConstraint = None\n"
        "    PhrasalConstraint = None\n"
        "try:\n"
        "    from transformers.generation.utils import GenerateOutput, SampleOutput, logger\n"
        "except ImportError:\n"
        "    from transformers.generation.utils import GenerateOutput, logger\n"
        "    SampleOutput = GenerateOutput"
    )

    patched = content.replace(old_import, new_import)
    if patched == content:
        print("stream_generator.py exact block not found")
    else:
        with open(stream_gen_path, "w", encoding="utf-8") as f:
            f.write(patched)
        print(f"Patched stream_generator.py: {stream_gen_path}")


def patch_tts_io():
    io_path = None
    for sp in site.getsitepackages() + sys.path:
        candidate = os.path.join(sp, "TTS/utils/io.py")
        if os.path.exists(candidate):
            io_path = candidate
            break

    if io_path is None:
        print("Could not find TTS/utils/io.py, skipping patch")
        return

    with open(io_path, "r", encoding="utf-8") as f:
        content = f.read()

    if "weights_only=False" in content:
        print("TTS/utils/io.py already patched")
        return

    old_load = "return torch.load(f, map_location=map_location, **kwargs)"
    new_load = "return torch.load(f, map_location=map_location, weights_only=False, **kwargs)"

    patched = content.replace(old_load, new_load)
    if patched == content:
        import re as _re

        patched = _re.sub(
            r"torch\.load\(([^)]+)\)",
            lambda m: m.group(0)
            if "weights_only" in m.group(0)
            else m.group(0).rstrip(")") + ", weights_only=False)",
            content,
        )

    if patched == content:
        print("Could not find torch.load call to patch")
    else:
        with open(io_path, "w", encoding="utf-8") as f:
            f.write(patched)
        print(f"Patched TTS/utils/io.py: {io_path}")


def patch_gpt2_inference_model():
    try:
        from transformers import GenerationMixin, PreTrainedModel
        from TTS.tts.layers.xtts.gpt import GPT2InferenceModel

        if issubclass(GPT2InferenceModel, GenerationMixin):
            print("GPT2InferenceModel already has GenerationMixin")
            return

        GPT2InferenceModel.__bases__ = (PreTrainedModel, GenerationMixin)
        if hasattr(GPT2InferenceModel, "generate"):
            print("Patched GPT2InferenceModel successfully")
        else:
            print("Patch applied but .generate() is still missing")

    except ImportError as e:
        print(f"Could not patch GPT2InferenceModel: {e}")
    except Exception as e:
        print(f"Unexpected error patching GPT2InferenceModel: {e}")


patch_stream_generator()
patch_tts_io()

mods_to_drop = [k for k in sys.modules if "TTS" in k or "transformers" in k]
for mod in mods_to_drop:
    sys.modules.pop(mod, None)
print(f"Cleared {len(mods_to_drop)} cached modules")

from TTS.api import TTS

patch_gpt2_inference_model()

BASE_WORK_DIR = Path("/kaggle/working/nbmmoshi")
INPUT_FILE = BASE_WORK_DIR / "augmented" / "augmented_full.jsonl"
OUTPUT_DIR = BASE_WORK_DIR / "speech_data"
WAV_DIR = OUTPUT_DIR / "wavs"
TRANS_DIR = OUTPUT_DIR / "transcripts"
SAMPLE_RATE = 24000

WAV_DIR.mkdir(parents=True, exist_ok=True)
TRANS_DIR.mkdir(parents=True, exist_ok=True)


def number_to_words_simple(text: str) -> str:
    try:
        from num2words import num2words

        def replace_num(match):
            num_str = match.group(0).replace(",", "")
            try:
                return num2words(float(num_str)) if "." in num_str else num2words(int(num_str))
            except Exception:
                return match.group(0)

        return re.sub(r"-?\d[\d,]*\.?\d*", replace_num, text)
    except ImportError:
        return text


print("Loading TTS model (Coqui XTTS-v2)...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")
print("TTS model loaded successfully")

USER_SPEAKERS = ["Claribel Dervla", "Daisy Studious", "Gracie Wise", "Tammie Ema", "Alison Dietlinde"]
ASSISTANT_SPEAKER = "Andrew Chipper"


def synthesize_speech(text: str, speaker: str) -> np.ndarray:
    spoken_text = number_to_words_simple(text)
    wav = tts.tts(text=spoken_text, speaker=speaker, language="en")
    return np.array(wav, dtype=np.float32)


def make_stereo_wav(assistant_audio: np.ndarray, user_audio: np.ndarray) -> np.ndarray:
    max_len = max(len(assistant_audio), len(user_audio))
    if len(assistant_audio) < max_len:
        assistant_audio = np.pad(assistant_audio, (0, max_len - len(assistant_audio)))
    if len(user_audio) < max_len:
        user_audio = np.pad(user_audio, (0, max_len - len(user_audio)))
    stereo = np.stack([assistant_audio, user_audio], axis=-1)
    stereo = stereo / (np.abs(stereo).max() + 1e-8) * 0.9
    return (stereo * 32767).astype(np.int16)


with open(INPUT_FILE, encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

print(f"Processing {len(data)} samples...")
dataset_index = []
error_count = 0

for i, item in enumerate(data):
    try:
        user_speaker = USER_SPEAKERS[i % len(USER_SPEAKERS)]
        user_audio = synthesize_speech(item["question"], user_speaker)
        assistant_audio = synthesize_speech(item["generated_concise_answer"], ASSISTANT_SPEAKER)

        stereo = make_stereo_wav(assistant_audio, user_audio)
        wav_filename = f"{i:05d}.wav"
        wav_path = WAV_DIR / wav_filename
        wavfile.write(wav_path, SAMPLE_RATE, stereo)

        transcript = {
            "id": item["id"],
            "wav_file": wav_filename,
            "user_text": item["question"],
            "assistant_text": item["generated_concise_answer"],
            "reasoning_text": item["generated_reasoning"],
            "ground_truth": item["ground_truth"],
            "duration_seconds": round(len(stereo) / SAMPLE_RATE, 2),
        }

        trans_path = TRANS_DIR / f"{i:05d}.json"
        with open(trans_path, "w", encoding="utf-8") as tf:
            json.dump(transcript, tf)

        dataset_index.append(
            {
                "audio": str(wav_path),
                "transcript": str(trans_path),
                "duration": transcript["duration_seconds"],
            }
        )

    except Exception as e:
        error_count += 1
        print(f"ERROR on sample {i}: {e}")

    if (i + 1) % 50 == 0:
        print(f"[{i + 1}/{len(data)}] Done | Saved: {len(dataset_index)} | Errors: {error_count}")

index_path = OUTPUT_DIR / "dataset.jsonl"
with open(index_path, "w", encoding="utf-8") as f:
    for entry in dataset_index:
        f.write(json.dumps(entry) + "\n")

print("Part 4 complete.")
print(f"Saved {len(dataset_index)}/{len(data)} samples to {OUTPUT_DIR}")
print(f"Errors: {error_count}")


In [ ]:
# Part 5: Fine-tune Moshi
import json
from pathlib import Path

import torch
import torch.nn as nn
import torchaudio
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import DataLoader, Dataset
from transformers import BitsAndBytesConfig

from moshi.models import Mimi, Moshi
from moshi.tokenizers import SentencePieceTokenizer

BASE_WORK_DIR = Path("/kaggle/working/nbmmoshi")
INPUT_DATA_DIR = BASE_WORK_DIR / "speech_data"
BASE_MODEL_DIR = "kyutai/moshika-pytorch-q8"
OUTPUT_DIR = BASE_WORK_DIR / "moshi_finetuned"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


class InnerMonologueDataset(Dataset):
    def __init__(self, data_dir, tokenizer, target_sample_rate=24000):
        self.data = []
        self.tokenizer = tokenizer
        self.sr = target_sample_rate

        with open(Path(data_dir) / "dataset.jsonl", "r", encoding="utf-8") as f:
            for line in f:
                entry = json.loads(line)
                entry["audio_path"] = entry["audio"]

                with open(entry["transcript"], encoding="utf-8") as tf:
                    entry["text_tokens"] = json.load(tf)["reasoning_text"]
                self.data.append(entry)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        entry = self.data[idx]
        wav, sr = torchaudio.load(entry["audio_path"])
        if sr != self.sr:
            wav = torchaudio.functional.resample(wav, sr, self.sr)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        text_ids = self.tokenizer.encode(entry["text_tokens"])
        return {"wav": wav.squeeze(0), "text_ids": torch.tensor(text_ids, dtype=torch.long)}


def collate_fn(batch):
    return {"wav": batch[0]["wav"], "text_ids": batch[0]["text_ids"].unsqueeze(0)}


device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading Mimi codec and 8-bit Moshi backbone...")
mimi = Mimi.from_pretrained(BASE_MODEL_DIR).to(device)
mimi.eval().requires_grad_(False)

moshi = Moshi.from_pretrained(
    BASE_MODEL_DIR,
    quantization_config=BitsAndBytesConfig(load_in_8bit=True, llm_int8_threshold=6.0),
    device_map="auto",
)
moshi = prepare_model_for_kbit_training(moshi)

lora_config = LoraConfig(
    r=128,
    lora_alpha=256,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
moshi = get_peft_model(moshi, lora_config)

tokenizer = SentencePieceTokenizer.from_pretrained(BASE_MODEL_DIR)
dataset = InnerMonologueDataset(INPUT_DATA_DIR, tokenizer)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

optimizer = torch.optim.AdamW(moshi.parameters(), lr=5e-5)
text_criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_id)
audio_criterion = nn.CrossEntropyLoss()

moshi.train()
print("Starting native dual-stream training...")

for epoch in range(5):
    for step, batch in enumerate(dataloader):
        wavs = batch["wav"].unsqueeze(0).to(device)
        text_ids = batch["text_ids"].to(device)
        optimizer.zero_grad()

        with torch.no_grad():
            audio_codes = mimi.encode(wavs.unsqueeze(1))

        outputs = moshi(
            text_input_ids=text_ids,
            audio_input_codes=audio_codes[:, :, :-1],
            return_dict=True,
        )

        text_logits = outputs.text_logits
        text_loss = text_criterion(
            text_logits[:, :-1, :].reshape(-1, text_logits.size(-1)),
            text_ids[:, 1:].reshape(-1),
        )

        audio_logits = outputs.audio_logits
        audio_loss = sum(
            audio_criterion(
                audio_logits[:, q, :, :].reshape(-1, audio_logits.size(-1)),
                audio_codes[:, q, 1:].reshape(-1),
            )
            for q in range(audio_logits.shape[1])
        )

        loss = text_loss + audio_loss
        loss.backward()
        optimizer.step()

        if step % 10 == 0:
            print(f"Epoch {epoch} | Step {step} | Total Loss: {loss.item():.4f}")

    save_path = OUTPUT_DIR / f"moshi_cot_epoch_{epoch}"
    moshi.save_pretrained(save_path)
    print(f"Saved checkpoint to {save_path}")


In [ ]:
# Part 6: Evaluate the fine-tuned model
import json
import re
from pathlib import Path

import torch
import torchaudio
import whisper
from peft import PeftModel

from moshi.models import Mimi, Moshi

BASE_WORK_DIR = Path("/kaggle/working/nbmmoshi")
INPUT_TEST_FILE = BASE_WORK_DIR / "speech_data" / "dataset.jsonl"
BASE_MODEL_DIR = "kyutai/moshika-pytorch-q8"
EVAL_OUTPUT_DIR = BASE_WORK_DIR / "evaluation"
LORA_PATH = BASE_WORK_DIR / "moshi_finetuned" / "moshi_cot_epoch_4"

EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def extract_number(text: str) -> str | None:
    numbers = re.findall(r"-?\d+\.?\d*", text.replace(",", ""))
    return numbers[-1] if numbers else None


def evaluate_native_moshi(is_lora: bool, test_data: list, system_name: str):
    print(f"Evaluating: {system_name}")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    mimi = Mimi.from_pretrained(BASE_MODEL_DIR).to(device).eval()
    moshi = Moshi.from_pretrained(BASE_MODEL_DIR, load_in_8bit=True, device_map="auto")

    if is_lora:
        moshi = PeftModel.from_pretrained(moshi, LORA_PATH)
    moshi.eval()

    whisper_model = whisper.load_model("large-v3").to(device)
    results = []

    for i, item in enumerate(test_data):
        try:
            wav, sr = torchaudio.load(item["audio"])
            wav = torchaudio.functional.resample(wav, sr, 24000)
            user_input_wav = wav[1].unsqueeze(0).to(device)

            output_audio_path = EVAL_OUTPUT_DIR / f"eval_{system_name}_{i}.wav"

            with torch.no_grad():
                input_codes = mimi.encode(user_input_wav.unsqueeze(1))
                generated_codes = moshi.generate(audio_input_codes=input_codes, max_new_tokens=500)
                output_wav = mimi.decode(generated_codes)

            torchaudio.save(str(output_audio_path), output_wav.squeeze(1).cpu(), 24000)

            predicted_text = whisper_model.transcribe(str(output_audio_path))["text"]

            with open(item["transcript"], encoding="utf-8") as tf:
                ground_truth = json.load(tf)["ground_truth"]

            pred_num = extract_number(predicted_text)
            correct = pred_num is not None and abs(float(pred_num) - float(ground_truth)) < 1e-2
            results.append({"predicted_text": predicted_text, "correct": correct})

        except Exception as e:
            print(f"ERROR on sample {i}: {e}")
            results.append({"correct": False})

    acc = sum(1 for r in results if r["correct"]) / len(results) * 100
    print(f"{system_name} Accuracy: {acc:.1f}%")
    return results


with open(INPUT_TEST_FILE, encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f][:10]

evaluate_native_moshi(is_lora=True, test_data=test_data, system_name="Moshi_With_CoT")


In [ ]:
!ls /kaggle/input/datasets/thedevastator/grade-school-math-8k-q-a/